# AI Stock Prediction - XGBoost Training
**Mục tiêu:** Huấn luyện model XGBoost không bị Data Leakage (loại bỏ hoàn toàn giá tuyệt đối).

In [ ]:
# === BƯỚC 1: KẾT NỐI GOOGLE DRIVE VÀ LOAD DATA ===
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
import pickle, json, os

df = pd.read_csv('/content/drive/MyDrive/StockData/all_stocks_train.csv')
df['trading_date'] = pd.to_datetime(df['trading_date'])
df = df.sort_values(['symbol', 'trading_date']).reset_index(drop=True)

print(f"Raw data: {len(df):,} dòng, {df['symbol'].nunique()} mã")

In [ ]:
# === BƯỚC 2: FILTER MÃ CHẤT LƯỢNG ===
# Chỉ giữ HOSE và HNX, bỏ UPCOM nhỏ
df = df[df['exchange'].isin(['HOSE', 'HNX'])]

# Chỉ giữ mã có thanh khoản tốt (> 100k cp/phiên)
df = df[df['volume_sma_20'] > 100000]

print(f"Sau filter: {len(df):,} dòng, {df['symbol'].nunique()} mã")

In [ ]:
# === BƯỚC 3: TẠO FEATURES MỚI VÀ CHUẨN HOÁ ===
df = df.sort_values(['symbol', 'trading_date'])

# 1. Normalized Trend & MA
df['price_vs_sma20'] = (df['close'] - df['sma_20']) / df['sma_20']
df['price_vs_sma50'] = (df['close'] - df['sma_50']) / df['sma_50']
df['sma20_vs_sma50'] = (df['sma_20'] - df['sma_50']) / df['sma_50']

# 2. Normalized Momentum (Price & Volume)
df['price_momentum_5']  = df.groupby('symbol')['close'].pct_change(5)
df['price_momentum_10'] = df.groupby('symbol')['close'].pct_change(10)
df['price_momentum_20'] = df.groupby('symbol')['close'].pct_change(20)
df['volume_momentum_5'] = df.groupby('symbol')['volume'].pct_change(5)

# 3. Normalized MACD & ATR & Bollinger
# Thêm một hằng số nhỏ (1e-6) để tránh lỗi chia cho 0
df['macd_norm'] = df['macd'] / (df['close'] + 1e-6)
df['macd_hist_norm'] = df['macd_hist'] / (df['close'] + 1e-6)
df['atr_pct'] = df['atr_14'] / (df['close'] + 1e-6)
df['bb_width_norm'] = df['bb_width'] / (df['sma_20'] + 1e-6)

# 4. Normalized Candlestick shape (so với giá open)
df['candle_body_pct'] = df['candle_body_size'] / (df['open'] + 1e-6)
df['candle_upper_pct'] = df['candle_upper_wick'] / (df['open'] + 1e-6)
df['candle_lower_pct'] = df['candle_lower_wick'] / (df['open'] + 1e-6)

# Clip các giá trị % cực đoan
momentum_cols = [
    'price_momentum_5', 'price_momentum_10', 'price_momentum_20', 'volume_momentum_5',
    'price_vs_sma20', 'price_vs_sma50', 'sma20_vs_sma50',
    'macd_norm', 'macd_hist_norm', 'atr_pct', 'bb_width_norm',
    'candle_body_pct', 'candle_upper_pct', 'candle_lower_pct'
]
for col in momentum_cols:
    df[col] = df[col].clip(-0.5, 0.5) # Giới hạn biến động ±50%

print("Đã tạo xong features mới đã chuẩn hoá hoàn toàn")

In [ ]:
# === BƯỚC 4: ĐỊNH NGHĨA FEATURES MỚI ===
FEATURES = [
    # Trend (Tuyệt đối xoá toàn bộ MA nguyên thuỷ)
    'price_vs_sma20', 'price_vs_sma50', 'sma20_vs_sma50',

    # Momentum
    'price_momentum_5', 'price_momentum_10', 'price_momentum_20',

    # MACD (Normalized)
    'macd_norm', 'macd_hist_norm',

    # Trend strength (0-100)
    'adx_14', 'di_plus', 'di_minus',

    # Oscillators (Bounded)
    'rsi_14', 'stoch_k', 'stoch_d', 'williams_r',

    # Bollinger Bands (Normalized)
    'bb_pct_b', 'bb_width_norm',

    # Volatility (Normalized)
    'atr_pct',

    # Volume (Chỉ dùng tương đối)
    'cmf_20', 'volume_ratio', 'volume_momentum_5',

    # Candlestick patterns (Normalized)
    'candle_body_pct', 'candle_upper_pct', 'candle_lower_pct',
    'is_doji', 'is_hammer',
    'is_bull_engulfing', 'is_bear_engulfing',
    'is_morning_star', 'is_evening_star'
]

TARGET = 'direction_5d'
print(f"Tổng features: {len(FEATURES)}")

In [ ]:
# === BƯỚC 5: LÀM SẠCH DATA ===
# Bỏ cột NaN > 50%
nan_pct = df[FEATURES].isna().mean()
bad_cols = nan_pct[nan_pct > 0.5].index.tolist()
if bad_cols:
    print(f"Bỏ features NaN > 50%: {bad_cols}")
    FEATURES = [f for f in FEATURES if f not in bad_cols]

# Bỏ dòng thiếu target
df = df.dropna(subset=[TARGET])
df[TARGET] = df[TARGET].astype(int)

# Forward fill NaN trong features theo từng mã
df[FEATURES] = df.groupby('symbol')[FEATURES].ffill()

# Bỏ dòng còn NaN sau fill
df = df.dropna(subset=FEATURES)

print(f"Sau làm sạch: {len(df):,} dòng, {df['symbol'].nunique()} mã")
print(f"Tỷ lệ tăng: {df[TARGET].mean():.2%}")

In [ ]:
# === BƯỚC 6: TÁCH TRAIN/TEST THEO THỜI GIAN ===
# TUYỆT ĐỐI không shuffle — đây là time series
CUTOFF = '2025-10-01'
train = df[df['trading_date'] <  CUTOFF]
test  = df[df['trading_date'] >= CUTOFF]

X_train = train[FEATURES]
y_train = train[TARGET]
X_test  = test[FEATURES]
y_test  = test[TARGET]

print(f"\nTrain: {len(X_train):,} dòng")
print(f"  Period: {train['trading_date'].min().date()} → {train['trading_date'].max().date()}")
print(f"  Tỷ lệ tăng: {y_train.mean():.2%}")
print(f"\nTest: {len(X_test):,} dòng")
print(f"  Period: {test['trading_date'].min().date()} → {test['trading_date'].max().date()}")
print(f"  Tỷ lệ tăng: {y_test.mean():.2%}")

In [ ]:
# === BƯỚC 7: TRAIN XGBOOST ===
# Xử lý mất cân bằng
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"\nscale_pos_weight = {scale_pos_weight:.2f}")

model = xgb.XGBClassifier(
    n_estimators          = 1000,
    max_depth             = 5,
    learning_rate         = 0.05,
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    min_child_weight      = 10,
    gamma                 = 0.2,
    reg_alpha             = 0.1,
    reg_lambda            = 1.0,
    scale_pos_weight      = scale_pos_weight,
    eval_metric           = 'auc',
    early_stopping_rounds = 50,
    random_state          = 42,
    n_jobs                = -1,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=100
)

In [ ]:
# === BƯỚC 8: ĐÁNH GIÁ ===
y_pred      = model.predict(X_test)
y_pred_prob = model.predict_proba(X_test)[:, 1]

print("\n" + "="*60)
print("KẾT QUẢ MODEL MỚI")
print("="*60)
print(f"Accuracy : {accuracy_score(y_test, y_pred):.2%}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_pred_prob):.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Giảm', 'Tăng']))

# Win rate theo từng ngưỡng
print("\nWin rate theo ngưỡng:")
print(f"{'Ngưỡng':>8} {'Số tín hiệu':>12} {'Win rate':>10} {'Avg return':>12}")
print("-" * 46)
test_eval = test.copy()
test_eval['score'] = y_pred_prob
for threshold in [0.50, 0.55, 0.60, 0.65, 0.70]:
    signals = test_eval[test_eval['score'] >= threshold]
    if len(signals) == 0:
        continue
    win_rate   = (signals[TARGET] == 1).mean()
    avg_return = signals['return_5d'].mean() if 'return_5d' in signals else 0
    print(f"{threshold:>8.2f} {len(signals):>12,} {win_rate:>10.2%} {avg_return:>12.2%}")

# Feature importance
fi = pd.DataFrame({
    'feature':    FEATURES,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 15 features quan trọng nhất:")
print(fi.head(15).to_string(index=False))

plt.figure(figsize=(10, 8))
plt.barh(fi.head(15)['feature'][::-1], fi.head(15)['importance'][::-1])
plt.title('Feature Importance — Model Mới')
plt.tight_layout()
plt.show()

# Kiểm tra nhanh trên các mã đang bị sai
print("\nKiểm tra mã PLX và MCH (đang bị model cũ dự báo sai):")
test_check = test[test['symbol'].isin(['PLX', 'MCH'])].copy()
if len(test_check) > 0:
    test_check['score'] = model.predict_proba(test_check[FEATURES])[:, 1]
    print(test_check[['symbol', 'trading_date', 'score', 'rsi_14', 'macd_norm', TARGET]].tail(10).to_string())

In [ ]:
# === BƯỚC 9: LƯU MODEL ===
OUTPUT_DIR = '/content/drive/MyDrive/StockData/models'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Lưu model
with open(f'{OUTPUT_DIR}/xgb_direction_5d.pkl', 'wb') as f:
    pickle.dump(model, f)

# Lưu features — QUAN TRỌNG: Đã được làm sạch, không còn giá nguyên thuỷ
with open(f'{OUTPUT_DIR}/features.json', 'w') as f:
    json.dump({'features': FEATURES}, f, indent=2)

# Lưu feature importance
fi.to_csv(f'{OUTPUT_DIR}/feature_importance.csv', index=False)

print("✅ Đã lưu thành công:")
print(f"   Model: {OUTPUT_DIR}/xgb_direction_5d.pkl")
print(f"   Features: {OUTPUT_DIR}/features.json")
print(f"   Feature importance: {OUTPUT_DIR}/feature_importance.csv")
print(f"\nDanh sách {len(FEATURES)} features đã lưu:")
print(FEATURES)